First try for the project : data extraction

In [ ]:
#dataset 1

import requests
import pandas as pd
from pathlib import Path

# Période d'étude
START_DATE = "2025-01-01"
END_DATE = "2025-09-01"

# Coordonnées de Paris
LATITUDE = 48.8566
LONGITUDE = 2.3522

# Dossier des données brutes
DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m",
        "wind_direction_10m",
        "surface_pressure"
    ],
    "timezone": "UTC"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

# Sauvegarde de la réponse JSON brute
with open(DATA_DIR / "meteo.json", "wb") as f:
    f.write(response.content)

# Conversion en DataFrame pour l'exploration
df_meteo = pd.DataFrame(response.json()["hourly"])

print("Dimensions :", df_meteo.shape)
display(df_meteo.head())

Dimensions : (168, 7)


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,surface_pressure
0,2025-09-01T00:00,14.8,89,0.0,8.7,198,1003.6
1,2025-09-01T01:00,15.2,91,0.1,8.2,195,1004.2
2,2025-09-01T02:00,15.0,92,0.1,8.0,207,1003.7
3,2025-09-01T03:00,14.6,93,0.0,8.5,215,1003.7
4,2025-09-01T04:00,14.3,93,0.0,8.1,215,1003.5


In [3]:
url = (
    "https://opendata.paris.fr/api/explore/v2.1/"
    "catalog/datasets/comptages-routiers-permanents/exports/csv"
)

params = {
    "where": (
        f"t_1h >= date'{START_DATE}' "
        f"AND t_1h < date'{pd.Timestamp(END_DATE) + pd.Timedelta(days=1):%Y-%m-%d}'"
    ),
    "timezone": "UTC",
    "delimiter": ";"
}

response = requests.get(url, params=params, timeout=120)
response.raise_for_status()

# Sauvegarde du CSV brut
with open(DATA_DIR / "trafic.csv", "wb") as f:
    f.write(response.content)

# Chargement dans Pandas
df_trafic = pd.read_csv(DATA_DIR / "trafic.csv", sep=";")

print("Dimensions :", df_trafic.shape)
display(df_trafic.head())

Dimensions : (491535, 15)


,iu_ac,libelle,t_1h,q,k,etat_trafic,iu_nd_amont,libelle_nd_amont,iu_nd_aval,libelle_nd_aval,etat_barre,date_debut,date_fin,geo_point_2d,geo_shape
0,1397,Av_Republique,2025-09-07T23:00:00+00:00,141.0,1.93611,Fluide,749,Av_Republique-St_Maur,702,Republique-Parmentier-Oberkampf,Ouvert,2005-01-01,2019-06-01,"48.86523164123044, 2.3749533513181817","{""coordinates"": [[2.375315267628968, 48.865167..."
1,1398,Av_Republique,2025-09-07T23:00:00+00:00,140.0,1.17722,Fluide,702,Republique-Parmentier-Oberkampf,749,Av_Republique-St_Maur,Ouvert,1996-10-03,2023-01-01,"48.86513482542217, 2.3748513636427395","{""coordinates"": [[2.3746890058208203, 48.86516..."
2,4301,Av_des_Ternes,2025-09-07T23:00:00+00:00,443.0,1.65056,Fluide,2310,Bd_Pereire-Av_Ternes,2313,Ternes-Gouvion,Ouvert,1996-10-10,2023-01-01,"48.88046890241474, 2.2866820449829186","{""coordinates"": [[2.2879804156937067, 48.88017..."
3,4305,Av_des_Ternes,2025-09-07T23:00:00+00:00,NaN,NaN,Inconnu,2315,Pl_Tristan_Bernard,2310,Bd_Pereire-Av_Ternes,Invalide,1996-10-10,2023-01-01,"48.879741179909985, 2.2897685612519183","{""coordinates"": [[2.291076421865212, 48.879433..."
4,4309,Av_des_Ternes,2025-09-07T23:00:00+00:00,330.0,NaN,Inconnu,2315,Pl_Tristan_Bernard,2316,Av_Ternes-Av_Niel,Ouvert,1996-10-10,2023-01-01,"48.878930291694004, 2.2926027939588325","{""coordinates"": [[2.2910226214597853, 48.87929..."


In [4]:
print("========== MÉTÉO ==========")

print("Dimensions :", df_meteo.shape)
print("\nColonnes :", df_meteo.columns.tolist())
print("\nTypes :")
print(df_meteo.dtypes)
print("\nValeurs manquantes :")
print(df_meteo.isna().sum())

print("\n========== TRAFIC ==========")

print("Dimensions :", df_trafic.shape)
print("\nColonnes :", df_trafic.columns.tolist())
print("\nTypes :")
print(df_trafic.dtypes)
print("\nValeurs manquantes :")
print(df_trafic.isna().sum())

========== MÉTÉO ==========
Dimensions : (168, 7)

Colonnes : ['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'wind_direction_10m', 'surface_pressure']

Types :
time                        str
temperature_2m          float64
relative_humidity_2m      int64
precipitation           float64
wind_speed_10m          float64
wind_direction_10m        int64
surface_pressure        float64
dtype: object

Valeurs manquantes :
time                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
wind_direction_10m      0
surface_pressure        0
dtype: int64

========== TRAFIC ==========
Dimensions : (491535, 15)

Colonnes : ['iu_ac', 'libelle', 't_1h', 'q', 'k', 'etat_trafic', 'iu_nd_amont', 'libelle_nd_amont', 'iu_nd_aval', 'libelle_nd_aval', 'etat_barre', 'date_debut', 'date_fin', 'geo_point_2d', 'geo_shape']

Types :
iu_ac                 int64
libelle                 str
t_1h                  